In [1]:
## Baseline model

import json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

def load_splits(path='dataset_splits.json'):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def evaluate_10fold_cv(make_pipeline_fn, cv_df, text_col, test_texts, y_test,
                      n_folds=10):
    """
    Ported from reference notebook. Fits TF-IDF inside each fold.
    """
    fold_accs = []
    fold_f1s = []

    for fold_idx in range(n_folds):
        val_mask = cv_df["fold"] == fold_idx
        train_mask = ~val_mask

        X_tr = cv_df.loc[train_mask, text_col].tolist()
        y_tr = cv_df.loc[train_mask, "label_id"].values
        X_val = cv_df.loc[val_mask, text_col].tolist()
        y_val = cv_df.loc[val_mask, "label_id"].values

        pipe = make_pipeline_fn()
        pipe.fit(X_tr, y_tr)
        val_preds = pipe.predict(X_val)

        fold_accs.append(accuracy_score(y_val, val_preds))
        fold_f1s.append(f1_score(y_val, val_preds, average="macro"))

    # Final fit on all CV data to evaluate on test set
    final_pipe = make_pipeline_fn()
    X_cv_all = cv_df[text_col].tolist()
    y_cv_all = cv_df["label_id"].values
    final_pipe.fit(X_cv_all, y_cv_all)

    test_preds = final_pipe.predict(test_texts)
    test_acc = accuracy_score(y_test, test_preds)
    test_f1 = f1_score(y_test, test_preds, average="macro")

    return {
        "cv_acc_mean": np.mean(fold_accs),
        "cv_acc_std": np.std(fold_accs),
        "cv_f1_mean": np.mean(fold_f1s),
        "cv_f1_std": np.std(fold_f1s),
        "test_acc": test_acc,
        "test_f1": test_f1,
        "report": classification_report(y_test, test_preds, target_names=["no", "maybe", "yes"])
    }

def make_baseline_pipe():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True, stop_words="english",
            ngram_range=(1, 2), max_features=30000
        )),
        ("lr", LogisticRegression(
            class_weight="balanced", random_state=42, max_iter=1000
        ))
    ])

def run_baseline():
    splits = load_splits()
    cv_df = pd.DataFrame(splits['cv'])
    test_df = pd.DataFrame(splits['test'])

    y_test = test_df["label_id"].values
    X_test_qctx = test_df["q_ctx"].tolist()

    print("============================================================")
    print("BASELINE: TF-IDF (1,2)-grams + Logistic Regression")
    print("Protocol: 10-Fold CV on 500 samples, Test on 500 samples")
    print("============================================================")

    results = evaluate_10fold_cv(
        make_baseline_pipe,
        cv_df,
        "q_ctx",
        X_test_qctx,
        y_test
    )

    print(f"10-Fold CV Accuracy : {results['cv_acc_mean']:.4f} ± {results['cv_acc_std']:.4f}")
    print(f"10-Fold CV Macro-F1 : {results['cv_f1_mean']:.4f} ± {results['cv_f1_std']:.4f}")
    print(f"Test Accuracy       : {results['test_acc']:.4f}")
    print(f"Test Macro-F1       : {results['test_f1']:.4f}")
    print("\nTest Classification Report:")
    print(results['report'])

if __name__ == "__main__":
    run_baseline()


BASELINE: TF-IDF (1,2)-grams + Logistic Regression
Protocol: 10-Fold CV on 500 samples, Test on 500 samples
10-Fold CV Accuracy : 0.5160 ± 0.0488
10-Fold CV Macro-F1 : 0.3432 ± 0.0525
Test Accuracy       : 0.5060
Test Macro-F1       : 0.3482

Test Classification Report:
              precision    recall  f1-score   support

          no       0.40      0.33      0.36       169
       maybe       0.13      0.04      0.06        55
         yes       0.56      0.71      0.63       276

    accuracy                           0.51       500
   macro avg       0.37      0.36      0.35       500
weighted avg       0.46      0.51      0.47       500



In [2]:
import os
import json
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm

def load_data(path='dataset_splits.json'):
    """Loads the official dataset splits."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dataset splits file not found at {path}")

    with open(path, 'r', encoding='utf-8') as f:
        splits = json.load(f)

    cv_df = pd.DataFrame(splits['cv'])
    test_df = pd.DataFrame(splits['test'])
    return cv_df, test_df

def get_embeddings(texts, model_name, batch_size=16):
    """
    Extracts frozen embeddings using a Transformer model and mean pooling.
    This replaces SentenceTransformer functionality if the library is missing.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc=f"Encoding with {model_name}"):
        batch_texts = texts[i:i+batch_size]
        # Same MAX_LEN as the rest of the project
        encoded_input = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=384,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            model_output = model(**encoded_input)

        # Perform Mean Pooling (Standard for sentence embeddings)
        attention_mask = encoded_input['attention_mask']
        token_embeddings = model_output[0] # Last hidden state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_embeddings = sum_embeddings / sum_mask

        all_embeddings.append(mean_embeddings.cpu().numpy())

    return np.vstack(all_embeddings)

def run_biobert_lr_experiment():
    """
    Implements A1d: BioBERT Embeddings + LR.
    Frozen representations are used with a Logistic Regression classifier.
    """
    print("--- BioBERT Embeddings + Logistic Regression (A1d) ---")

    cv_df, test_df = load_data()

    y_cv = cv_df["label_id"].values
    y_test = test_df["label_id"].values

    # Protocol A1d: Uses Question + Context
    X_cv_text = cv_df["q_ctx"].tolist()
    X_test_text = test_df["q_ctx"].tolist()

    # Using the specialized BioBERT model for sentence embeddings used in the group notebook
    MODEL_NAME = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"

    print("\nPhase 1: Embedding Extraction")
    X_cv_emb = get_embeddings(X_cv_text, MODEL_NAME)
    X_test_emb = get_embeddings(X_test_text, MODEL_NAME)

    print("\nPhase 2: 10-Fold Cross-Validation")
    fold_f1s = []

    for fold_idx in range(10):
        val_mask = cv_df["fold"] == fold_idx
        train_mask = ~val_mask

        X_tr, y_tr = X_cv_emb[train_mask], y_cv[train_mask]
        X_val, y_val = X_cv_emb[val_mask], y_cv[val_mask]

        # Consistent with report and notebook settings
        clf = LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        )
        clf.fit(X_tr, y_tr)
        preds = clf.predict(X_val)

        f1 = f1_score(y_val, preds, average="macro", zero_division=0)
        fold_f1s.append(f1)
        print(f"  Fold {fold_idx}: Macro-F1 = {f1:.4f}")

    print(f"\nCV Results: Mean Macro-F1 = {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}")

    print("\nPhase 3: Final Test Evaluation")
    final_clf = LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    )
    final_clf.fit(X_cv_emb, y_cv)
    test_preds = final_clf.predict(X_test_emb)

    print("\nFinal Report (A1d):")
    print(f"Test Accuracy: {accuracy_score(y_test, test_preds):.4f}")
    print(f"Test Macro-F1: {f1_score(y_test, test_preds, average='macro', zero_division=0):.4f}")
    print("\nDetailed Classification Report:")
    print(classification_report(
        y_test, test_preds,
        target_names=["no", "maybe", "yes"],
        zero_division=0
    ))

if __name__ == "__main__":
    run_biobert_lr_experiment()


--- BioBERT Embeddings + Logistic Regression (A1d) ---

Phase 1: Embedding Extraction
Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Encoding with pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb: 100%|██████████| 32/32 [00:14<00:00,  2.23it/s]


Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Encoding with pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb: 100%|██████████| 32/32 [00:13<00:00,  2.40it/s]



Phase 2: 10-Fold Cross-Validation
  Fold 0: Macro-F1 = 0.3475
  Fold 1: Macro-F1 = 0.3428
  Fold 2: Macro-F1 = 0.3130
  Fold 3: Macro-F1 = 0.4140
  Fold 4: Macro-F1 = 0.3974
  Fold 5: Macro-F1 = 0.3845
  Fold 6: Macro-F1 = 0.3356
  Fold 7: Macro-F1 = 0.4080
  Fold 8: Macro-F1 = 0.4340
  Fold 9: Macro-F1 = 0.2601

CV Results: Mean Macro-F1 = 0.3637 ± 0.0508

Phase 3: Final Test Evaluation

Final Report (A1d):
Test Accuracy: 0.5140
Test Macro-F1: 0.4185

Detailed Classification Report:
              precision    recall  f1-score   support

          no       0.45      0.44      0.44       169
       maybe       0.16      0.20      0.18        55
         yes       0.64      0.62      0.63       276

    accuracy                           0.51       500
   macro avg       0.42      0.42      0.42       500
weighted avg       0.52      0.51      0.52       500



In [3]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
from collections import Counter
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score, classification_report


RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# load data
def load_data():
    with open('dataset_splits.json', 'r', encoding='utf-8') as f:
        splits = json.load(f)

    cv_df = pd.DataFrame(splits['cv'])
    test_df = pd.DataFrame(splits['test'])

    dev_fold_mask = cv_df["fold"] == 0
    train_fold_mask = ~dev_fold_mask

    # uses q_ctx
    train_texts = cv_df.loc[train_fold_mask, "q_ctx"].tolist()
    train_labels = cv_df.loc[train_fold_mask, "label_id"].values.tolist()

    val_texts = cv_df.loc[dev_fold_mask, "q_ctx"].tolist()
    val_labels = cv_df.loc[dev_fold_mask, "label_id"].values.tolist()

    test_texts = test_df["q_ctx"].tolist()
    test_labels = test_df["label_id"].values.tolist()

    return train_texts, train_labels, val_texts, val_labels, test_texts, test_labels

# Dataset and weights
class QADataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=384):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item

def get_class_weights(labels):
    counts = Counter(labels)
    total = len(labels)
    n_classes = 3
    weights = [total / (n_classes * counts[i]) for i in range(n_classes)]
    return torch.tensor(weights, dtype=torch.float32).to(device)

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").to(model.device)
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }


# Run training
def run_finetuning():
    MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"
    MAX_LEN = 384
    BATCH_SIZE = 8
    EPOCHS = 8
    LEARNING_RATE = 2e-5

    tr_txt, tr_lbl, val_txt, val_lbl, ts_txt, ts_lbl = load_data()
    weights = get_class_weights(tr_lbl)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)

    train_ds = QADataset(tr_txt, tr_lbl, tokenizer, MAX_LEN)
    val_ds = QADataset(val_txt, val_lbl, tokenizer, MAX_LEN)
    test_ds = QADataset(ts_txt, ts_lbl, tokenizer, MAX_LEN)

    steps_per_epoch = max(1, len(train_ds) // BATCH_SIZE)
    total_steps = steps_per_epoch * EPOCHS
    warmup_steps = int(0.1 * total_steps)

    training_args = TrainingArguments(
        output_dir="./biobert_output",
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        weight_decay=0.01,
        warmup_steps=warmup_steps,
        report_to="none",
        fp16=True,
        save_total_limit=2
    )

    trainer = WeightedTrainer(
        class_weights=weights,
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    print("\nStarting BioBERT Fine-tuning (Identical to Notebook)...")
    trainer.train()

    print("\nEvaluating on Test Set...")
    test_results = trainer.predict(test_ds)
    preds = np.argmax(test_results.predictions, axis=1)

    print("\nFinal Results:")
    print(f"Accuracy: {accuracy_score(ts_lbl, preds):.4f}")
    print(f"Macro F1: {f1_score(ts_lbl, preds, average='macro'):.4f}")
    print("\nClassification Report:")
    print(classification_report(ts_lbl, preds, target_names=["no", "maybe", "yes"]))

if __name__ == "__main__":
    run_finetuning()

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne


Starting BioBERT Fine-tuning (Identical to Notebook)...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.136242,1.089457,0.520000,0.256971
2,1.078886,1.074731,0.480000,0.350255
3,0.892730,1.175162,0.460000,0.319022
4,0.635094,1.197351,0.480000,0.361416
5,0.506395,1.451761,0.460000,0.326667
6,0.336634,1.564156,0.480000,0.334522


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Evaluating on Test Set...



Final Results:
Accuracy: 0.3700
Macro F1: 0.3102

Classification Report:
              precision    recall  f1-score   support

          no       0.35      0.67      0.46       169
       maybe       0.11      0.16      0.13        55
         yes       0.64      0.23      0.34       276

    accuracy                           0.37       500
   macro avg       0.37      0.35      0.31       500
weighted avg       0.48      0.37      0.36       500



In [4]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
from collections import Counter
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score, classification_report


RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# Load data
def load_data():
    with open('dataset_splits.json', 'r', encoding='utf-8') as f:
        splits = json.load(f)

    cv_df = pd.DataFrame(splits['cv'])
    test_df = pd.DataFrame(splits['test'])

    dev_fold_mask = cv_df["fold"] == 0
    train_fold_mask = ~dev_fold_mask

    # uses q_ctx_ans
    train_texts = cv_df.loc[train_fold_mask, "q_ctx_ans"].tolist()
    train_labels = cv_df.loc[train_fold_mask, "label_id"].values.tolist()

    val_texts = cv_df.loc[dev_fold_mask, "q_ctx_ans"].tolist()
    val_labels = cv_df.loc[dev_fold_mask, "label_id"].values.tolist()

    test_texts = test_df["q_ctx_ans"].tolist()
    test_labels = test_df["label_id"].values.tolist()

    return train_texts, train_labels, val_texts, val_labels, test_texts, test_labels


# Dataset and weights
class QADataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=384):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item

def get_class_weights(labels):
    counts = Counter(labels)
    total = len(labels)
    n_classes = 3
    weights = [total / (n_classes * counts[i]) for i in range(n_classes)]
    return torch.tensor(weights, dtype=torch.float32).to(device)

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").to(model.device)
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

# Run training
def run_finetuning():
    MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"
    MAX_LEN = 384
    BATCH_SIZE = 8
    EPOCHS = 8
    LEARNING_RATE = 2e-5

    tr_txt, tr_lbl, val_txt, val_lbl, ts_txt, ts_lbl = load_data()
    weights = get_class_weights(tr_lbl)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)

    train_ds = QADataset(tr_txt, tr_lbl, tokenizer, MAX_LEN)
    val_ds = QADataset(val_txt, val_lbl, tokenizer, MAX_LEN)
    test_ds = QADataset(ts_txt, ts_lbl, tokenizer, MAX_LEN)

    steps_per_epoch = max(1, len(train_ds) // BATCH_SIZE)
    total_steps = steps_per_epoch * EPOCHS
    warmup_steps = int(0.1 * total_steps)

    training_args = TrainingArguments(
        output_dir="./biobert_output",
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        weight_decay=0.01,
        warmup_steps=warmup_steps,
        report_to="none",
        fp16=True,
        save_total_limit=2
    )

    trainer = WeightedTrainer(
        class_weights=weights,
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    print("\nStarting BioBERT Fine-tuning (Identical to Notebook)...")
    trainer.train()

    print("\nEvaluating on Test Set...")
    test_results = trainer.predict(test_ds)
    preds = np.argmax(test_results.predictions, axis=1)

    print("\nFinal Results:")
    print(f"Accuracy: {accuracy_score(ts_lbl, preds):.4f}")
    print(f"Macro F1: {f1_score(ts_lbl, preds, average='macro'):.4f}")
    print("\nClassification Report:")
    print(classification_report(ts_lbl, preds, target_names=["no", "maybe", "yes"]))

if __name__ == "__main__":
    run_finetuning()


Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne


Starting BioBERT Fine-tuning (Identical to Notebook)...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.125913,1.076881,0.520000,0.368254
2,0.969255,1.083313,0.640000,0.412553
3,0.831072,1.010035,0.620000,0.525395
4,0.544941,1.095374,0.580000,0.487606
5,0.380062,1.113402,0.520000,0.449303


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Evaluating on Test Set...



Final Results:
Accuracy: 0.5800
Macro F1: 0.4403

Classification Report:
              precision    recall  f1-score   support

          no       0.49      0.83      0.61       169
       maybe       0.15      0.05      0.08        55
         yes       0.76      0.53      0.63       276

    accuracy                           0.58       500
   macro avg       0.47      0.47      0.44       500
weighted avg       0.60      0.58      0.56       500

